# Etapa 1 (RAW completo): filtro MG por ponto-poligono

Objetivo desta etapa:
- usar o catalogo RAW completo (sem filtro previo por ST);
- decidir manter/remover evento usando apenas coordenadas e limite oficial de MG;
- auditar inconsistencias entre a tag ST e a geometria.


In [1]:
## 1) Checagem de dependencias

import importlib.util
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import display

# Este notebook usa `geobr` em runtime para obter o limite de MG.
import geobr, geopandas, shapely

## 2) Configuracao e leitura do RAW completo


In [18]:
raw_relpath = Path("catalogs/sisbra/sisbra_v2024May09/catalogo_RAW_v2024May09.csv")
start_dir = Path.cwd().resolve()
search_roots = [start_dir, *start_dir.parents]

repo_root = None
for candidate in search_roots:
    if (candidate / raw_relpath).exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        f"Nao encontrei {raw_relpath} a partir de {start_dir}. Execute no repo ou ajuste o caminho."
    )

raw_csv = repo_root / raw_relpath
df_raw = pd.read_csv(raw_csv)

print("repo_root:", repo_root)
print("arquivo RAW:", raw_csv)
print("linhas RAW:", len(df_raw))
print("colunas RAW:", len(df_raw.columns))


repo_root: /home/gabrielgoes/projetos/ClassificadorSismologico
arquivo RAW: /home/gabrielgoes/projetos/ClassificadorSismologico/catalogs/sisbra/sisbra_v2024May09/catalogo_RAW_v2024May09.csv
linhas RAW: 5934
colunas RAW: 19


## 3) Padronizacao minima

Nesta etapa:
- normalizamos `ST` para auditoria;
- convertimos `latit` e `longit` para numerico;
- marcamos quais linhas possuem coordenada valida.


In [19]:
df = df_raw.copy()

required_cols = ["latit", "longit", "ST"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Colunas obrigatorias ausentes no RAW: {missing_cols}")

df["ST_norm"] = df["ST"].fillna("").astype(str).str.strip().str.upper()
df["latit_num"] = pd.to_numeric(df["latit"], errors="coerce")
df["longit_num"] = pd.to_numeric(df["longit"], errors="coerce")

df["has_valid_latlon"] = (
    df["latit_num"].between(-90, 90, inclusive="both")
    & df["longit_num"].between(-180, 180, inclusive="both")
)

print("linhas com coordenada valida:", int(df["has_valid_latlon"].sum()))
print("linhas sem coordenada valida:", int((~df["has_valid_latlon"]).sum()))


linhas com coordenada valida: 5790
linhas sem coordenada valida: 144


## 4) Limite de MG em runtime (deterministico)

Regra desta etapa: se nao conseguir baixar o limite oficial de MG, interromper com erro claro.


In [20]:
import geobr

try:
    mg_state = geobr.read_state(code_state="MG", year=2020)
except Exception as exc:
    raise RuntimeError(
        "Falha ao carregar limite de MG via geobr. Verifique internet/servico e tente novamente."
    ) from exc

if mg_state.empty:
    raise RuntimeError("geobr retornou geometria vazia para MG.")

if mg_state.crs is None:
    mg_state = mg_state.set_crs("EPSG:4674")

#mg_polygon = mg_state.geometry.unary_union
mg_polygon = mg_state.geometry.union_all()
if mg_polygon.is_empty:
    raise RuntimeError("Geometria de MG ficou vazia apos uniao do poligono.")

print("CRS limite MG:", mg_state.crs)
print("Limite MG carregado com sucesso.")


CRS limite MG: EPSG:4674
Limite MG carregado com sucesso.


## 5) Classificacao deterministica: evento dentro de MG? (sim/nao)


In [21]:
df["inside_mg_polygon"] = False

df_valid = df[df["has_valid_latlon"]].copy()
points = gpd.GeoDataFrame(
    df_valid,
    geometry=gpd.points_from_xy(df_valid["longit_num"], df_valid["latit_num"]),
    crs="EPSG:4326",
)

if points.crs != mg_state.crs:
    points = points.to_crs(mg_state.crs)

inside_mask = points.geometry.intersects(mg_polygon)
df.loc[df_valid.index, "inside_mg_polygon"] = inside_mask.to_numpy()

print("eventos dentro de MG (com coordenada valida):", int(df["inside_mg_polygon"].sum()))
print(
    "eventos fora de MG (com coordenada valida):",
    int((df["has_valid_latlon"] & ~df["inside_mg_polygon"]).sum()),
)


eventos dentro de MG (com coordenada valida): 918
eventos fora de MG (com coordenada valida): 4872


## 6) Status do filtro MG

- `KEEP_IN_MG`: ponto dentro de MG;
- `DROP_OUTSIDE_MG`: ponto fora de MG;
- `DROP_NO_VALID_COORDS`: sem coordenada valida.


In [22]:
df["mg_filter_status"] = "DROP_OUTSIDE_MG"
df.loc[~df["has_valid_latlon"], "mg_filter_status"] = "DROP_NO_VALID_COORDS"
df.loc[df["has_valid_latlon"] & df["inside_mg_polygon"], "mg_filter_status"] = "KEEP_IN_MG"

df_in_mg = df[df["mg_filter_status"] == "KEEP_IN_MG"].copy()
df_out_mg = df[df["mg_filter_status"] == "DROP_OUTSIDE_MG"].copy()
df_no_coords = df[df["mg_filter_status"] == "DROP_NO_VALID_COORDS"].copy()

print("KEEP_IN_MG:", len(df_in_mg))
print("DROP_OUTSIDE_MG:", len(df_out_mg))
print("DROP_NO_VALID_COORDS:", len(df_no_coords))


KEEP_IN_MG: 918
DROP_OUTSIDE_MG: 4872
DROP_NO_VALID_COORDS: 144


## 7) Auditoria ST x geometria

Aqui a tag ST e apenas auditoria de consistencia humana contra regra geometrica.


In [23]:
df["st_geo_consistency"] = "NOT_APPLICABLE_NO_COORDS"

valid_mask = df["has_valid_latlon"]
st_empty = df["ST_norm"].eq("")
st_mg = df["ST_norm"].eq("MG")
inside = df["inside_mg_polygon"]

df.loc[valid_mask & st_empty, "st_geo_consistency"] = "UNKNOWN_ST"
df.loc[valid_mask & ~st_empty & st_mg & inside, "st_geo_consistency"] = "CONSISTENT_ST_MG_INSIDE"
df.loc[valid_mask & ~st_empty & st_mg & ~inside, "st_geo_consistency"] = "INCONSISTENT_ST_MG_OUTSIDE"
df.loc[valid_mask & ~st_empty & ~st_mg & inside, "st_geo_consistency"] = "INCONSISTENT_ST_NOT_MG_INSIDE"
df.loc[valid_mask & ~st_empty & ~st_mg & ~inside, "st_geo_consistency"] = "CONSISTENT_ST_NOT_MG_OUTSIDE"

print(df["st_geo_consistency"].value_counts(dropna=False).to_string())


st_geo_consistency
CONSISTENT_ST_NOT_MG_OUTSIDE     4851
CONSISTENT_ST_MG_INSIDE           905
NOT_APPLICABLE_NO_COORDS          144
INCONSISTENT_ST_MG_OUTSIDE         15
INCONSISTENT_ST_NOT_MG_INSIDE      13
UNKNOWN_ST                          6


## 8) Tabelas prioritarias de inconsistencias

- Caso critico 1: `ST=MG` mas ponto fora de MG.
- Caso critico 2: `ST!=MG` (inclui vazio) mas ponto dentro de MG.


In [24]:
cols_show = [
    "year", "mm", "dd", "hh", "min", "ss.s",
    "ST", "ST_norm", "latit", "longit", "CAT", "mag", "tm", "Localities",
    "inside_mg_polygon", "mg_filter_status", "st_geo_consistency",
]

for col in cols_show:
    if col not in df.columns:
        df[col] = pd.NA

case_st_mg_outside = df[
    df["has_valid_latlon"] & df["ST_norm"].eq("MG") & ~df["inside_mg_polygon"]
].copy()

case_st_not_mg_inside = df[
    df["has_valid_latlon"] & ~df["ST_norm"].eq("MG") & df["inside_mg_polygon"]
].copy()

case_st_empty_inside = df[
    df["has_valid_latlon"] & df["ST_norm"].eq("") & df["inside_mg_polygon"]
].copy()

print("Inconsistencia A (ST=MG fora de MG):", len(case_st_mg_outside))
if len(case_st_mg_outside) > 0:
    display(case_st_mg_outside[cols_show].head(30))

print("Inconsistencia B (ST!=MG dentro de MG):", len(case_st_not_mg_inside))
if len(case_st_not_mg_inside) > 0:
    display(case_st_not_mg_inside[cols_show].head(30))

print("Subconjunto B1 (ST vazio dentro de MG):", len(case_st_empty_inside))
if len(case_st_empty_inside) > 0:
    display(case_st_empty_inside[cols_show].head(30))


Inconsistencia A (ST=MG fora de MG): 15


,year,mm,dd,hh,min,ss.s,ST,ST_norm,latit,longit,CAT,mag,tm,Localities,inside_mg_polygon,mg_filter_status,st_geo_consistency
393,1950,2.0,27.0,8.0,58.0,NaN,MG,MG,-21.82,-46.71,R,3.9,3,P. DE CALDAS,False,DROP_OUTSIDE_MG,INCONSISTENT_ST_MG_OUTSIDE
806,1982,5.0,2.0,8.0,30.0,4.0,MG,MG,-21.64,-46.65,A,3.1,1,P. DE CALDAS,False,DROP_OUTSIDE_MG,INCONSISTENT_ST_MG_OUTSIDE
899,1984,7.0,4.0,16.0,7.0,5.0,MG,MG,-20.20,-47.35,I,2.4,1,ESTREITO,False,DROP_OUTSIDE_MG,INCONSISTENT_ST_MG_OUTSIDE
2936,2014,7.0,21.0,19.0,50.0,12.0,MG,MG,-18.91,-40.98,I,1.3,5,Mantena,False,DROP_OUTSIDE_MG,INCONSISTENT_ST_MG_OUTSIDE
3131,2015,4.0,2.0,2.0,42.0,19.0,MG,MG,-21.02,-42.07,I,1.9,1,Antonio Prado,False,DROP_OUTSIDE_MG,INCONSISTENT_ST_MG_OUTSIDE
3183,2015,6.0,1.0,15.0,19.0,0.0,MG,MG,-23.58,-51.86,I,2.9,1,Marialva,False,DROP_OUTSIDE_MG,INCONSISTENT_ST_MG_OUTSIDE
3311,2015,11.0,13.0,19.0,37.0,15.0,MG,MG,-9.22,-36.01,I,2.2,1,Branquinha,False,DROP_OUTSIDE_MG,INCONSISTENT_ST_MG_OUTSIDE
3413,2016,2.0,12.0,2.0,10.0,4.0,MG,MG,-22.04,-40.75,I,1.9,1,-,False,DROP_OUTSIDE_MG,INCONSISTENT_ST_MG_OUTSIDE
3498,2016,3.0,29.0,10.0,9.0,57.0,MG,MG,-23.69,-42.68,I,2.0,1,-,False,DROP_OUTSIDE_MG,INCONSISTENT_ST_MG_OUTSIDE
3579,2016,7.0,2.0,23.0,55.0,22.0,MG,MG,-23.59,-41.79,I,1.8,1,-,False,DROP_OUTSIDE_MG,INCONSISTENT_ST_MG_OUTSIDE


Inconsistencia B (ST!=MG dentro de MG): 13


,year,mm,dd,hh,min,ss.s,ST,ST_norm,latit,longit,CAT,mag,tm,Localities,inside_mg_polygon,mg_filter_status,st_geo_consistency
413,1957,NaN,NaN,NaN,NaN,NaN,SP,SP,-20.30,-49.20,C,0.0,-1,ICEM,True,KEEP_IN_MG,INCONSISTENT_ST_NOT_MG_INSIDE
843,1983,7.0,12.0,20.0,43.0,57.0,GO,GO,-18.70,-49.20,I,2.7,1,CENTRALINA,True,KEEP_IN_MG,INCONSISTENT_ST_NOT_MG_INSIDE
1580,1995,6.0,3.0,5.0,26.0,31.0,GO,GO,-18.03,-46.74,I,3.1,1,Vazante,True,KEEP_IN_MG,INCONSISTENT_ST_NOT_MG_INSIDE
1644,1996,4.0,27.0,14.0,47.0,12.0,MT,MT,-18.12,-43.69,I,3.2,1,Alto Garcas,True,KEEP_IN_MG,INCONSISTENT_ST_NOT_MG_INSIDE
1754,1997,7.0,17.0,16.0,45.0,45.0,SP,SP,-20.82,-47.17,I,2.9,1,Antas,True,KEEP_IN_MG,INCONSISTENT_ST_NOT_MG_INSIDE
2053,2002,10.0,3.0,14.0,7.0,42.0,SP,SP,-21.84,-46.65,I,2.6,1,S.J.BoaVista,True,KEEP_IN_MG,INCONSISTENT_ST_NOT_MG_INSIDE
2155,2004,12.0,27.0,14.0,24.0,5.0,SP,SP,-22.23,-44.80,I,2.2,1,Cruzeiro,True,KEEP_IN_MG,INCONSISTENT_ST_NOT_MG_INSIDE
2910,2014,6.0,21.0,17.0,6.0,48.0,SP,SP,-19.99,-47.26,I,2.9,1,Rifaina,True,KEEP_IN_MG,INCONSISTENT_ST_NOT_MG_INSIDE
3443,2016,2.0,26.0,20.0,1.0,7.0,PE,PE,-22.24,-45.92,I,1.7,5,Pouso Alegre,True,KEEP_IN_MG,INCONSISTENT_ST_NOT_MG_INSIDE
3806,2017,6.0,22.0,2.0,25.0,29.0,GO,GO,-16.01,-47.28,I,2.4,1,divisa GO-MG,True,KEEP_IN_MG,INCONSISTENT_ST_NOT_MG_INSIDE


Subconjunto B1 (ST vazio dentro de MG): 0


## 9) Resumo final da etapa


In [25]:
summary_rows = [
    {"metric": "rows_raw_total", "value": int(len(df))},
    {"metric": "rows_keep_in_mg", "value": int((df["mg_filter_status"] == "KEEP_IN_MG").sum())},
    {"metric": "rows_drop_outside_mg", "value": int((df["mg_filter_status"] == "DROP_OUTSIDE_MG").sum())},
    {"metric": "rows_drop_no_valid_coords", "value": int((df["mg_filter_status"] == "DROP_NO_VALID_COORDS").sum())},
    {"metric": "incons_st_mg_outside", "value": int(len(case_st_mg_outside))},
    {"metric": "incons_st_not_mg_inside", "value": int(len(case_st_not_mg_inside))},
    {"metric": "incons_st_empty_inside", "value": int(len(case_st_empty_inside))},
]

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


,metric,value
0,rows_raw_total,5934
1,rows_keep_in_mg,918
2,rows_drop_outside_mg,4872
3,rows_drop_no_valid_coords,144
4,incons_st_mg_outside,15
5,incons_st_not_mg_inside,13
6,incons_st_empty_inside,0
